In [ ]:
# Import des librairies importantesy
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import datetime

In [ ]:
# 1. Chargement & Préparation des données
df = pd.read_csv("cleaned_dataset.csv")
df = df.dropna(subset=["Average delay of all trains at arrival"])
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Hour"] = df["Date"].dt.hour

In [ ]:
# 2. Sélection des features pertinentes
features = ["Departure station", "Arrival station", "DayOfWeek", "Hour"]
target = "Average delay of all trains at arrival"

df_model = df[features + [target]].dropna()
X = pd.get_dummies(df_model[features])
y = df_model[target]

In [ ]:
# 3. Split des données
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# 4. Entraînement de modèles de base
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results[name] = {"rmse": rmse, "r2": r2}

In [ ]:
# 5. Tuning d'hyperparamètres pour Random Forest
param_grid = {"n_estimators": [100, 200], "max_depth": [10, 20, None]}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

# Meilleur modèle Random Forest
best_rf = grid_search.best_estimator_
best_rf_pred = best_rf.predict(X_test)
best_rf_rmse = np.sqrt(mean_squared_error(y_test, best_rf_pred))
best_rf_r2 = r2_score(y_test, best_rf_pred)

results["Best Random Forest (Tuned)"] = {"rmse": best_rf_rmse, "r2": best_rf_r2}

In [ ]:
# 6. Affichage des résultats
print("\n📊 Résultats des modèles :")
for name, res in results.items():
    print(f"{name:30s} | RMSE: {res['rmse']:.2f} | R²: {res['r2']:.2f}")

In [ ]:
# 7. Justification du meilleur modèle
best_model = max(results.items(), key=lambda x: x[1]["r2"])
print(
    f"\n✅ Le meilleur modèle est : {best_model[0]} avec un R² de {best_model[1]['r2']:.2f}"
)

In [ ]:
# Fonction pour prédire un retard à partir d’une requête utilisateur


def predict_delay(departure, arrival, dayofweek=None, hour=None):
    now = datetime.datetime.now()
    if dayofweek is None:
        dayofweek = now.weekday()
    if hour is None:
        hour = now.hour
    input_df = pd.DataFrame(
        [
            {
                "Departure_station": departure,
                "Arrival_station": arrival,
                "DayOfWeek": dayofweek,
                "Hour": hour,
            }
        ]
    )
    input_encoded = pd.get_dummies(input_df)
    missing_cols = set(X_train.columns) - set(input_encoded.columns)
    for col in missing_cols:
        input_encoded[col] = 0
    input_encoded = input_encoded[X_train.columns]
    prediction = model.predict(input_encoded)[0]
    return prediction